# 03b — EDA Stage 2: ADR City vs Resort

Cùng trình tự EDA ADR của notebook **03**, nhưng **tách và so sánh** hai property.

**Phạm vi:** `is_canceled = 0` và `adr > 0`.

1. Snapshot phân bố ADR
2. Arrival month (box, line overlay, heatmap năm)
3. Day of week
4. Room type + room_match
5. Customer type
6. Gap City − Resort theo tháng


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

NOTEBOOK_DIR = Path(os.environ.get("VSCODE_NOTEBOOK_DIR", Path.cwd()))
ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "data").is_dir() else NOTEBOOK_DIR
DATA_PATH = ROOT / "data" / "hotel_bookings_v5.csv"
FIG_DIR = ROOT / "reports" / "figures" / "03b"
FIG_DIR.mkdir(parents=True, exist_ok=True)

HOTELS = ["City Hotel", "Resort Hotel"]
HOTEL_COLORS = {"City Hotel": "#4C72B0", "Resort Hotel": "#55A868"}
MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BIN_LABELS = ["0-30", "31-60", "61-90", "91-180", ">180"]
DEPOSIT_ORDER = ["No Deposit", "Non Refund", "Refundable"]
ALPHA = 0.05

print(f"ROOT: {ROOT}")
print(f"DATA: {DATA_PATH}")
print(f"FIG_DIR: {FIG_DIR}")


In [ ]:
def fmt_int(n) -> str:
    return f"{int(round(n)):,}".replace(",", ".")

def fmt_pct(x: float, d: int = 1) -> str:
    return f"{x * 100:.{d}f}%".replace(".", ",")

def fmt_eur(x: float, d: int = 2) -> str:
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"{s} €"

def savefig(name: str) -> Path:
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    print(f"Saved: {path.relative_to(ROOT)}")
    return path

def add_lead_bin(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    edges = [0, 30, 60, 90, 180, float(out["lead_time"].max()) + 1]
    out["lead_time_bin"] = pd.cut(
        out["lead_time"], bins=edges, labels=BIN_LABELS, right=True, include_lowest=True
    )
    return out


In [ ]:
raw = pd.read_csv(DATA_PATH)
stay = raw[(raw["is_canceled"]==0) & (raw["adr"]>0)].copy()
stay["hotel"] = pd.Categorical(stay["hotel"], categories=HOTELS, ordered=True)
stay["arrival_date_month"] = pd.Categorical(stay["arrival_date_month"], categories=MONTH_ORDER, ordered=True)
stay["day_of_week"] = pd.Categorical(stay["day_of_week"], categories=DAY_ORDER, ordered=True)
stay["room_match"] = stay["reserved_room_type"] == stay["assigned_room_type"]
print(f"Stay ADR>0: {fmt_int(len(stay))}")
for h in HOTELS:
    g = stay[stay["hotel"]==h]
    print(f"  {h}: n={fmt_int(len(g))} | mean={fmt_eur(g['adr'].mean())} | median={fmt_eur(g['adr'].median())}")


## 0. Snapshot


In [ ]:
snap = (
    stay.groupby("hotel", observed=True)["adr"]
    .agg(bookings="count", mean="mean", median="median", std="std")
    .reindex(HOTELS)
)
display(snap.round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
sns.boxplot(data=stay, x="hotel", y="adr", hue="hotel", palette=HOTEL_COLORS, showfliers=False, ax=axes[0], legend=False)
for h in HOTELS:
    sns.kdeplot(stay.loc[stay["hotel"]==h, "adr"], ax=axes[1], label=h, color=HOTEL_COLORS[h], clip=(0,300))
axes[1].legend(); axes[0].set_title("Box ADR"); axes[1].set_title("KDE ADR")
savefig("00_snapshot.png"); plt.show()


## 1. Arrival month


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
month_rows = []
for ax, h in zip(axes, HOTELS):
    g = stay[stay["hotel"]==h]
    sns.boxplot(data=g, x="arrival_date_month", y="adr", showfliers=False, ax=ax, color=HOTEL_COLORS[h])
    ax.set_title(h); ax.tick_params(axis="x", rotation=25); ax.set_xlabel("")
    t = g.groupby("arrival_date_month", observed=True)["adr"].agg(bookings="count", mean="mean", median="median", std="std").reindex(MONTH_ORDER).reset_index()
    t["hotel"]=h; month_rows.append(t)
savefig("01_monthly_box.png"); plt.show()
month = pd.concat(month_rows, ignore_index=True)
display(month.pivot(index="arrival_date_month", columns="hotel", values="mean").round(2))

fig, ax = plt.subplots(figsize=(11,5))
for h in HOTELS:
    t = month[month["hotel"]==h]
    ax.plot(t["arrival_date_month"].astype(str), t["mean"], marker="o", label=h, color=HOTEL_COLORS[h])
ax.legend(); ax.tick_params(axis="x", rotation=25); ax.set_title("Mean ADR theo thang")
savefig("02_monthly_mean_overlay.png"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
for ax, h in zip(axes, HOTELS):
    mat = stay[stay["hotel"]==h].pivot_table(index="arrival_date_month", columns="arrival_date_year", values="adr", aggfunc="mean", observed=True).reindex(MONTH_ORDER)
    sns.heatmap(mat, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax); ax.set_title(h)
savefig("03_heatmap_month_year.png"); plt.show()


## 2. Day of week


In [ ]:
dow = (
    stay.groupby(["hotel","day_of_week"], observed=True)["adr"]
    .agg(bookings="count", mean="mean", median="median").reset_index()
)
display(dow.pivot(index="day_of_week", columns="hotel", values="mean").reindex(DAY_ORDER).round(2))
fig, ax = plt.subplots(figsize=(11,5))
sns.barplot(data=dow, x="day_of_week", y="mean", hue="hotel", hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.set_title("Mean ADR theo day_of_week")
savefig("04_dow_bar.png"); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, h in zip(axes, HOTELS):
    sns.boxplot(data=stay[stay["hotel"]==h], x="day_of_week", y="adr", showfliers=False, ax=ax, color=HOTEL_COLORS[h])
    ax.set_title(h); ax.set_xlabel("")
savefig("05_dow_box.png"); plt.show()


## 3. Room type


In [ ]:
room = stay.groupby(["hotel","reserved_room_type"])["adr"].agg(bookings="count", mean="mean", median="median").reset_index()
display(room.pivot(index="reserved_room_type", columns="hotel", values="mean").round(2))
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=room, y="reserved_room_type", x="mean", hue="hotel", hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.set_title("Mean ADR theo reserved_room_type")
savefig("06_room_bar.png"); plt.show()

match = stay.groupby(["hotel","room_match"])["adr"].agg(bookings="count", mean="mean", median="median").reset_index()
display(match)
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.boxplot(data=stay, x="hotel", y="adr", hue="room_match", showfliers=False, ax=ax)
ax.set_title("ADR theo room_match")
savefig("07_room_match_box.png"); plt.show()

mat = stay.pivot_table(index="reserved_room_type", columns="hotel", values="adr", aggfunc="mean")
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(mat, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
ax.set_title("Mean ADR room x hotel")
savefig("08_heatmap_room_hotel.png"); plt.show()


## 4. Customer type + gap


In [ ]:
cust = stay.groupby(["hotel","customer_type"])["adr"].agg(bookings="count", mean="mean", median="median").reset_index()
display(cust.pivot(index="customer_type", columns="hotel", values="mean").round(2))
fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=cust, x="customer_type", y="mean", hue="hotel", hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.set_title("Mean ADR theo customer_type")
savefig("09_customer_bar.png"); plt.show()

fig, ax = plt.subplots(figsize=(11,5))
sns.boxplot(data=stay, x="customer_type", y="adr", hue="hotel", hue_order=HOTELS, palette=HOTEL_COLORS, showfliers=False, ax=ax)
savefig("10_customer_box.png"); plt.show()

month = (
    stay.groupby(["hotel","arrival_date_month"], observed=True)["adr"].mean().reset_index()
)
pivot = month.pivot(index="arrival_date_month", columns="hotel", values="adr")
pivot["gap"] = pivot["City Hotel"] - pivot["Resort Hotel"]
display(pivot.round(2))
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.bar(pivot.index.astype(str), pivot["gap"], color="#8172B3")
ax.axhline(0, color="black", lw=0.8)
ax.tick_params(axis="x", rotation=25)
ax.set_title("Gap mean ADR (City - Resort)")
savefig("11_gap_monthly.png"); plt.show()
snap.to_csv(FIG_DIR / "kpi_compare_city_resort.csv")
